In [5]:
#!/usr/bin/env python3
"""
Generate erroneous reads from barcodes (one per line) with a FIXED total number of reads R,
allocated across barcodes via a multinomial distribution, and a ground-truth index file.

Allocation:
- Let n = number of barcodes.
- Choose weights w. By default, w is uniform (1/n each).
- Sample counts (k_0..k_{n-1}) ~ Multinomial(R, w).
- For each barcode i, generate k_i independent erroneous reads.
- Ground truth file repeats the barcode index i exactly k_i times, aligned to reads.

Error model (iterative; based on the paper's Algorithm 2):
- Per-step mutation probability p.
- If mutation occurs, choose equally among substitution/insertion/deletion (p/3 each).
- Disallow insertion immediately after deletion and vice versa by redistributing their probability mass.

Usage:
  python generate_reads_multinomial.py barcodes.txt reads.txt truth.txt --p 0.2 --total-reads 1000000 --seed 42

Optional:
  --min-per-barcode 1   ensures every barcode gets at least 1 read (requires total_reads >= n)
"""

import argparse
import random
import sys
from typing import List

BASES: List[str] = ["A", "C", "G", "T"]


def rand_base(rng: random.Random) -> str:
    return rng.choice(BASES)


def rand_base_not(rng: random.Random, avoid: str) -> str:
    return rng.choice([b for b in BASES if b != avoid])


def simulate_read(barcode: str, p: float, rng: random.Random) -> str:
    """Simulate an erroneous read of fixed length len(barcode)."""
    if not (0.0 <= p <= 1.0):
        raise ValueError(f"p must be in [0,1], got {p}")

    n = len(barcode)
    psub = p / 3.0
    pins = p / 3.0
    pdel = p / 3.0

    p_ins_prime = pins
    p_del_prime = pdel

    i = 0  # index in barcode
    o = 0  # index in output read
    out = [""] * n

    while i < n and o < n:
        pnone = max(0.0, 1.0 - (psub + p_ins_prime + p_del_prime))
        r = rng.random()

        if r < psub:
            event = "sub"
        elif r < psub + p_ins_prime:
            event = "ins"
        elif r < psub + p_ins_prime + p_del_prime:
            event = "del"
        else:
            event = "none"

        if event == "sub":
            out[o] = rand_base_not(rng, barcode[i])
            i += 1
            o += 1
            p_ins_prime = pins
            p_del_prime = pdel

        elif event == "ins":
            out[o] = rand_base(rng)
            o += 1
            # after insertion: disallow deletion next; move deletion mass to insertion
            p_ins_prime = pins + pdel
            p_del_prime = 0.0

        elif event == "del":
            i += 1
            # after deletion: disallow insertion next; move insertion mass to deletion
            p_ins_prime = 0.0
            p_del_prime = pins + pdel

        else:  # none
            out[o] = barcode[i]
            i += 1
            o += 1
            p_ins_prime = pins
            p_del_prime = pdel

    # pad if output read shorter than n
    while o < n:
        out[o] = rand_base(rng)
        o += 1

    return "".join(out)


def multinomial_counts(total: int, probs: List[float], rng: random.Random) -> List[int]:
    """
    Sample counts from Multinomial(total, probs) without numpy.
    Uses sequential binomial draws (numerically stable and exact).
    """
    if total < 0:
        raise ValueError("total must be >= 0")
    if not probs:
        return []
    s = sum(probs)
    if s <= 0:
        raise ValueError("Sum of probs must be > 0")
    probs = [p / s for p in probs]

    counts = [0] * len(probs)
    remaining = total
    remaining_prob = 1.0

    # Sequentially draw count for each category except last
    for i in range(len(probs) - 1):
        if remaining == 0:
            break
        # Conditional probability for category i given not assigned to previous categories
        pi = probs[i] / remaining_prob if remaining_prob > 0 else 0.0
        pi = min(max(pi, 0.0), 1.0)
        # Binomial draw: number assigned to this category
        # random.Random has no binomial, so we use sum of Bernoullis for small remaining,
        # and a normal approximation would be inexact. We'll implement an exact draw using
        # Python's built-in random.choices for generality:
        #
        # BUT doing random.choices remaining times can be slow for huge totals.
        # Therefore, implement an exact binomial via inversion (works well for moderate n),
        # and fall back to Bernoulli sum when remaining is small.
        #
        # We'll use a simple Bernoulli sum if remaining <= 2000; otherwise use inversion.
        if remaining <= 2000:
            x = sum(1 for _ in range(remaining) if rng.random() < pi)
        else:
            # Binomial inversion method
            # Reference: classic cumulative probability summation.
            # This is exact but can be slow if pi near 0.5 and remaining huge.
            # For typical barcode allocations with large n and small pi, it's fast.
            q = 1.0 - pi
            # Start at 0 successes
            prob = q ** remaining
            cdf = prob
            u = rng.random()
            x = 0
            while u > cdf and x < remaining:
                # Update probability mass function iteratively:
                # P(X=x+1) = P(X=x) * (n-x)/(x+1) * p/q
                prob *= (remaining - x) / (x + 1) * (pi / q if q > 0 else 0.0)
                cdf += prob
                x += 1

        counts[i] = x
        remaining -= x
        remaining_prob -= probs[i]

    # Last category gets the remainder
    counts[-1] = remaining
    return counts


In [20]:
barcodes_in = "/user/gent/446/vsc44685/ScratchVO_dir/barcalling_review/barcodes/inhouse/bar_1in2_column_major.txt"
#barcodes_in = "/user/gent/446/vsc44685/ScratchVO_dir/barcalling_review/barcodes/inhouse/bar_1in1_column_major.txt"
reads_out = "/user/gent/446/vsc44685/ScratchVO_dir/barcalling_review/nextflow_project/test_data/quik/sim_reads/reads_p_0.2_1in2.txt"
truth_out = "/user/gent/446/vsc44685/ScratchVO_dir/barcalling_review/nextflow_project/test_data/quik/sim_reads/truth_p_0.2_1in2.txt"
total_reads = 1000000
p = 0.2
seed = 123
min_per_barcode = 0
with open(barcodes_in, "r", encoding="utf-8") as f:
    barcodes = [line.strip().upper() for line in f if line.strip()]

In [21]:
if total_reads < 0:
    raise ValueError("--total-reads must be >= 0")
if min_per_barcode < 0:
    raise ValueError("--min-per-barcode must be >= 0")

rng = random.Random(seed)
# Read barcodes
with open(barcodes_in, "r", encoding="utf-8") as f:
    barcodes = [line.strip().upper() for line in f if line.strip()]

if not barcodes:
    print("No barcodes found.", file=sys.stderr)
    sys.exit(1)

# Validate fixed length and alphabet
L = len(barcodes[0])
for i, b in enumerate(barcodes):
    if len(b) != L:
        raise ValueError(f"Barcode length mismatch at line {i}: expected {L}, got {len(b)}")
    bad = set(b) - set(BASES)
    if bad:
        raise ValueError(f"Invalid characters in barcode at line {i}: {bad}")

n = len(barcodes)
min_total = n * min_per_barcode
if total_reads < min_total:
    raise ValueError(f"--total-reads must be >= n*min-per-barcode = {min_total}")

# Uniform weights (can be extended to user-provided weights if needed)
probs = [1.0 / n] * n

# Allocate counts
remaining_total = total_reads - min_total
extra = multinomial_counts(remaining_total, probs, rng) if remaining_total > 0 else [0] * n
counts = [min_per_barcode + extra[i] for i in range(n)]

# Sanity check: fixed total
assert sum(counts) == total_reads

# Generate reads
with open(reads_out, "w", encoding="utf-8") as f_reads, \
        open(truth_out, "w", encoding="utf-8") as f_truth:
    for idx, barcode in enumerate(barcodes):
        k = counts[idx]
        for _ in range(k):
            read = simulate_read(barcode, p, rng)
            f_reads.write(read + "\n")
            f_truth.write(str(idx) + "\n")

